# Stage 5A — Broaden the historical data

**What:** Keep your verified January 15 snapshot and collect one complete day from each remaining month of 2025. Save each day separately and create a dataset manifest that pins all 12 files.

**Why:** One day cannot show how the model works on later dates. Multiple months give us a first chronological training and test set.

**Limit:** One day per month is still a learning sample, not every 311 request in 2025. We will build a full scheduled collector later. These public-data calls and small Drive files use free tools.

In [ ]:
# Cell 2 — Why: choose dates in advance and keep the source fixed.
from google.colab import drive
from pathlib import Path
from datetime import date, datetime, timedelta, timezone
import hashlib, json, requests, sys
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
jan_snapshot_id = "20260926T104802395974Z"
new_days = [f"2025-{month:02d}-15" for month in range(2, 13)]
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
fields = ["unique_key", "created_date", "closed_date", "complaint_type", "borough"]

session = requests.Session()
retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry))
print("New dates:", new_days)

In [ ]:
# Cell 3 — Why: fetch a full day and reject missing or duplicate rows.
def fetch_day(day):
    next_day = (date.fromisoformat(day) + timedelta(days=1)).isoformat()
    start, end = f"{day}T00:00:00", f"{next_day}T00:00:00"
    where = f"created_date >= '{start}' AND created_date < '{end}'"
    count_reply = session.get(url, params={"$select": "count(*) as n", "$where": where}, timeout=60)
    count_reply.raise_for_status()
    expected = int(count_reply.json()[0]["n"])
    if not 0 < expected <= 50000:
        raise ValueError(f"{day}: needs paging; tell me the count {expected}")

    params = {"$select": ",".join(fields), "$where": where,
              "$order": "created_date ASC, unique_key ASC", "$limit": expected}
    reply = session.get(url, params=params, timeout=120)
    reply.raise_for_status()
    rows = reply.json()
    keys = [row["unique_key"] for row in rows]
    if len(rows) != expected or len(keys) != len(set(keys)):
        raise ValueError(f"{day}: incomplete or duplicate data")
    if any(not start <= row["created_date"] < end for row in rows):
        raise ValueError(f"{day}: a row is outside the date range")
    return rows, params

In [ ]:
# Cell 4 — Why: save each day once, then safely reuse it on reruns.
def save_day(day):
    day_dir = root / "raw_by_day" / day
    raw_file, manifest_file = day_dir / "requests.jsonl", day_dir / "manifest.json"
    if manifest_file.exists():
        meta = json.loads(manifest_file.read_text())
        saved_bytes = raw_file.read_bytes()
        if hashlib.sha256(saved_bytes).hexdigest() != meta["sha256"]:
            raise ValueError(f"{day}: saved file changed")
        if len(saved_bytes.splitlines()) != meta["rows"]:
            raise ValueError(f"{day}: saved row count changed")
        return meta
    if day_dir.exists():
        raise ValueError(f"{day}: incomplete folder; ask me to inspect it")

    rows, params = fetch_day(day)
    raw_bytes = "".join(json.dumps(row, sort_keys=True) + "\n" for row in rows).encode()
    meta = {"day": day, "source": url, "query": params, "rows": len(rows),
            "sha256": hashlib.sha256(raw_bytes).hexdigest(),
            "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
            "python": sys.version.split()[0]}
    day_dir.mkdir(parents=True, exist_ok=False)
    raw_file.write_bytes(raw_bytes)
    manifest_file.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return meta

In [ ]:
# Cell 5 — Why: collect the planned months and show progress.
for day in new_days:
    meta = save_day(day)
    print(day, "saved or verified:", meta["rows"], "rows")

In [ ]:
# Cell 6 — Why: pin every source file used in the larger dataset.
jan_dir = root / "raw" / jan_snapshot_id
jan_meta = json.loads((jan_dir / "manifest.json").read_text())
jan_file = jan_dir / "requests.jsonl"
if hashlib.sha256(jan_file.read_bytes()).hexdigest() != jan_meta["sha256"]:
    raise ValueError("January snapshot changed")

parts = [{"day": "2025-01-15", "file": str(jan_file.relative_to(root)),
          "sha256": jan_meta["sha256"], "rows": jan_meta["row_count"]}]
for day in new_days:
    day_dir = root / "raw_by_day" / day
    meta = json.loads((day_dir / "manifest.json").read_text())
    raw_file = day_dir / "requests.jsonl"
    if hashlib.sha256(raw_file.read_bytes()).hexdigest() != meta["sha256"]:
        raise ValueError(f"{day}: partition changed")
    parts.append({"day": day, "file": str(raw_file.relative_to(root)),
                  "sha256": meta["sha256"], "rows": meta["rows"]})
print("Verified days:", len(parts))
print("Total rows:", sum(part["rows"] for part in parts))

In [ ]:
# Cell 7 — Why: save one ID for this exact collection of 12 files.
dataset_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
dataset_dir = root / "datasets" / dataset_id
dataset_dir.mkdir(parents=True, exist_ok=False)
dataset_manifest = {
    "dataset_id": dataset_id,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "sampling_rule": "all requests from the 15th of each month in 2025",
    "partitions": parts,
    "total_rows": sum(part["rows"] for part in parts),
    "lesson_version": "stage5a-v1",
}
manifest_path = dataset_dir / "manifest.json"
manifest_path.write_text(json.dumps(dataset_manifest, indent=2), encoding="utf-8")
print("Dataset ID:", dataset_id)
print("Dataset manifest:", manifest_path)

# Stage 5A takeaway and quiz

**Why this stage mattered:** We broadened our data across months and pinned every raw file with a checksum in one dataset manifest. That gives later training a repeatable input. The sampling rule is documented, so we do not mistake these 12 days for a full year.

**Easy quiz:**

1. Why did we save each month's day separately?
2. If Cell 5 stops halfway through, what does `save_day` do with the days already saved when we rerun it?
3. What does the dataset manifest contain?
4. Does one day per month represent every request in 2025?

Send me the `Verified days`, `Total rows`, and `Dataset ID` outputs, plus your answers. Stage 5B will create labels for this broader data, split by time, and compare a baseline with a first model.